## 01 · Erstes 1D-CNN in PyTorch – vom Signal zur Vorhersage

**Von der App zum Code.** In der App
[Zeitreihen-CNN-Labor](https://iludis.de/zeitreihen-cnn-labor.html) habt ihr gesehen,
was ein **1D-Filter** mit einem Zeitreihen-Signal macht: Er gleitet über die Zeitachse
und erzeugt eine **Feature Map** – genau wie der 2D-Filter im
[CNN Grayscale Trainer](https://iludis.de/CNN_sim/CNN_sim.html) über ein Bild gleitet.

**Der Bogen:** Ihr kennt bereits klassische Zeitreihenverfahren (SARIMAX, Prophet) und
das 2D-CNN für Bilder (`01_basicCNN_didaktisch.ipynb`). Dieses Notebook verbindet beides:
dieselbe Faltungs-Idee wie beim 2D-CNN, aber angewendet auf eine **einzelne Zeitachse**
statt auf Höhe und Breite eines Bildes.

**Datensatz:** [AirPassengers](https://www.kaggle.com/datasets/rakannimer/air-passengers) –
monatliche Passagierzahlen 1949–1960, ein Klassiker der Zeitreihenanalyse.

In [ ]:
# ============================================================
# Konfiguration (zentral & leicht anpassbar)
# Alle Stellschrauben an einem Ort -> kein Suchen mehr quer durch den Code.
# ============================================================

# Reproduzierbarkeit
SEED = 42

# Pfade
DATA_PATH = "Datasets/AirPassengers.csv"

# Zeitfenster
WINDOW_SIZE = 12    # letzte 12 Monate -> Vorhersage für den 13. Monat
TRAIN_SPLIT = 0.8   # Anteil der Fenster fürs Training (zeitlich, nicht zufällig!)

# Architektur (die eigentlichen Stellschrauben des CNN)
N_CHANNELS   = 1    # 1 Kanal: die Passagierzahl
CONV_FILTERS = 8    # Filter im Conv1d-Block
KERNEL_SIZE  = 3    # Breite des Faltungskerns (Zeitschritte)
POOL_SIZE    = 2    # Reduktionsfaktor durch MaxPool1d
FC_HIDDEN    = 32   # Neuronen in der voll verbundenen Schicht

# Training
LEARNING_RATE = 0.01
MAX_EPOCHS    = 300
PRINT_EVERY   = 50  # alle N Epochen den Loss ausgeben


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reproduzierbarkeit: alle sehen dieselben Zufallswerte und Visualisierungen
torch.manual_seed(SEED)
np.random.seed(SEED)

# Haus-Stil
DUNKELBLAU = "#1B3A5C"
MITTELBLAU = "#2E75B6"
plt.rcParams["font.family"]     = "DejaVu Sans"   # Arial-nah (Arial oft nicht installiert)
plt.rcParams["axes.titlecolor"] = DUNKELBLAU
plt.rcParams["figure.dpi"]      = 110


### 1 · Daten ansehen – eine Zeitreihe ist eine Zahlenfolge

Genau wie ein Graustufenbild eine Matrix aus Pixelwerten ist, ist eine Zeitreihe eine
**Folge von Zahlen entlang der Zeit**. Statt Höhe × Breite haben wir hier nur eine
Dimension: die Zeit.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["Month"] = pd.to_datetime(df["Month"])
passagiere = df["Passengers"].values.astype(np.float32).reshape(-1, 1)

print(f"Anzahl Monate: {len(passagiere)}")
print("Erste Werte (Passagiere in 1000):", passagiere[:6].flatten())

plt.figure(figsize=(9, 3))
plt.plot(df["Month"], passagiere, color=MITTELBLAU)
plt.title("AirPassengers – monatliche Passagierzahlen 1949–1960")
plt.xlabel("Jahr"); plt.ylabel("Passagiere (in 1000)")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 2 · Normalisierung – wie Graustufen zwischen 0 und 1

Beim 2D-CNN haben wir die Pixelwerte durch 16 geteilt, damit sie zwischen 0 und 1
liegen. Bei Zeitreihen machen wir dasselbe mit dem **`MinMaxScaler`**: Er staucht alle
Werte auf den Bereich 0–1. Das hilft dem Netz genauso, schneller und stabiler zu
lernen wie beim Bild.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
daten_norm = scaler.fit_transform(passagiere)

print("Wertebereich vorher:", passagiere.min(), "-", passagiere.max())
print("Wertebereich nachher:", daten_norm.min(), "-", daten_norm.max())


### 3 · Zeitfenster erzeugen

Ein CNN braucht eine feste Eingabegröße. Beim Bild war das die 8×8-Matrix. Bei der
Zeitreihe schneiden wir stattdessen gleitende **Fenster** heraus: Die letzten
`window_size` Monate sind die Eingabe, der nächste Monat ist das Ziel – die
Vorhersage entsteht, indem das Netz aus der Vergangenheit in die Zukunft schaut.

In [ ]:
def erstelle_fenster(daten, window_size):
    '''Zerlegt eine Zeitreihe in überlappende Fenster (X) und die jeweils
    darauffolgenden Werte (y).'''
    xs, ys = [], []
    for i in range(len(daten) - window_size):
        xs.append(daten[i:i + window_size])
        ys.append(daten[i + window_size])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X, y = erstelle_fenster(daten_norm, WINDOW_SIZE)

print(f"Form X: {X.shape}  (Fenster, Zeitschritte, Merkmale)")
print(f"Form y: {y.shape}  (Fenster, Zielwert)")

beispiel = 0
plt.figure(figsize=(5, 2))
plt.plot(range(WINDOW_SIZE), X[beispiel], "o-", color=MITTELBLAU, label="Eingabefenster")
plt.plot(WINDOW_SIZE, y[beispiel], "o", color="red", label="Zielwert")
plt.title(f"Ein Trainingsbeispiel (Fenster #{beispiel})")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()


### 4 · Trainings-/Testdaten – **zeitlich**, nicht zufällig!

**Wichtiger Unterschied zum 2D-CNN:** Bei den Ziffernbildern war `train_test_split`
mit zufälliger Mischung völlig in Ordnung – ein Bild einer 3 hat nichts mit dem
nächsten zu tun. Bei einer Zeitreihe wäre zufälliges Mischen ein Fehler: Wir würden
dem Modell erlauben, aus der **Zukunft** zu lernen, um die **Vergangenheit**
vorherzusagen. Deshalb teilen wir hier strikt der Zeit nach: die ersten 80 % der
Monate zum Trainieren, die letzten 20 % – die das Modell nie gesehen hat – zum
Testen.

In [ ]:
split = int(TRAIN_SPLIT * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Trainingsfenster: {len(X_train)}   Testfenster: {len(X_test)}")


### 5 · Umwandlung in Tensoren

Ein 1D-CNN erwartet die Form **(Fenster, Kanäle, Zeitschritte)** – analog zu
**(Bilder, Kanal, Höhe, Breite)** beim 2D-CNN. Wir haben 1 Kanal (die Passagierzahl)
und müssen die Achsen dafür vertauschen (`permute`).

In [ ]:
X_train = torch.from_numpy(X_train).permute(0, 2, 1)  # (Fenster, 1, 12)
X_test  = torch.from_numpy(X_test).permute(0, 2, 1)
y_train = torch.from_numpy(y_train)
y_test  = torch.from_numpy(y_test)

print(f"Form X_train: {tuple(X_train.shape)}  (Fenster, Kanal, Zeitschritte)")
print(f"Form y_train: {tuple(y_train.shape)}")


### 6 · Das 1D-CNN als `nn.Sequential`

Derselbe Aufbau wie beim 2D-CNN – nur mit `Conv1d` und `MaxPool1d` statt `Conv2d` und
`MaxPool2d`. Der Filter gleitet nicht mehr über Höhe *und* Breite, sondern nur noch
über die **Zeitachse**.

In [ ]:
# Groesse nach MaxPool1d(POOL_SIZE): WINDOW_SIZE -> /POOL_SIZE
pooled_len  = WINDOW_SIZE // POOL_SIZE
flatten_dim = CONV_FILTERS * pooled_len

net = nn.Sequential(
    nn.Conv1d(N_CHANNELS, CONV_FILTERS, kernel_size=KERNEL_SIZE, padding=KERNEL_SIZE // 2),  # 0: 1x12 -> 8x12
    nn.ReLU(),                                   # 1
    nn.MaxPool1d(POOL_SIZE),                     # 2: 8x12 -> 8x6
    nn.Flatten(),                                # 3: 8*6 = 48
    nn.Linear(flatten_dim, FC_HIDDEN),           # 4
    nn.ReLU(),                                   # 5
    nn.Linear(FC_HIDDEN, 1),                     # 6: 1 Vorhersagewert
)
print(net)

total = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"\nTrainierbare Parameter insgesamt: {total}")

# Conv-Layer ist Element 0. Anfangs-Kernel sichern (Kopie!) für Vorher/Nachher.
kernel_vorher = net[0].weight.detach().clone()


### 7 · Tensor-Formen Schritt für Schritt verfolgen

Wie beim 2D-CNN: ein Fenster durch das Netz schicken und nach jeder Stufe die Form
prüfen. Ein `nn.Sequential` ist auch hier **schneidbar**: `net[:k]`.

`1×12` → Faltung `8×12` → MaxPool `8×6` → flach `48` → `32` → `1`

In [ ]:
beispiel_fenster = X_train[0:1]   # Form (1, 1, 12): ein einzelnes Fenster

with torch.no_grad():
    nach_conv = net[:1](beispiel_fenster)   # nach Faltung (= net[0])
    nach_relu = net[:2](beispiel_fenster)   # nach ReLU
    nach_pool = net[:3](beispiel_fenster)   # nach MaxPool

print(f"{'Stufe':<24}{'Form (Kanal x Zeitschritte)'}")
print("-" * 52)
print(f"{'Eingabe':<24}{tuple(beispiel_fenster.shape[1:])}")
print(f"{'Faltung (Conv1d)':<24}{tuple(nach_conv.shape[1:])}")
print(f"{'nach ReLU':<24}{tuple(nach_relu.shape[1:])}")
print(f"{'nach MaxPool':<24}{tuple(nach_pool.shape[1:])}")
print(f"{'flach (Flatten)':<24}{(flatten_dim,)}")
print(f"{'Ausgabe (1 Wert)':<24}{(1,)}")


### 8 · Training

Wie beim 2D-CNN: Full-Batch, Adam, feste Epochenzahl. Der Verlust (`MSELoss`) misst
hier den **Abstand zwischen Vorhersage und echtem Wert** – nicht "richtig/falsch"
wie bei der Ziffernklassifikation, sondern "wie nah dran".

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)

verlauf = []
net.train()
for epoch in range(MAX_EPOCHS):
    optimizer.zero_grad()
    output = net(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    verlauf.append(loss.item())
    if (epoch + 1) % PRINT_EVERY == 0:
        print(f"Epoch {epoch + 1}/{MAX_EPOCHS}, Loss: {loss.item():.4f}")

plt.figure(figsize=(5, 2.5))
plt.plot(verlauf, color=DUNKELBLAU)
plt.title("Verlauf des Trainingsverlusts")
plt.xlabel("Epoche"); plt.ylabel("MSE-Verlust (normiert)")
plt.tight_layout(); plt.show()


### 9 · Evaluation – **Fehlermaß statt Trefferquote**

**Wichtiger Unterschied zum 2D-CNN:** Bei den Ziffern gab es eine `accuracy_score`
(richtig/falsch). Eine Passagierzahl-Vorhersage ist aber keine Klasse, sondern ein
Punkt auf einer Skala. Wir messen deshalb den durchschnittlichen Fehler in der
**Originalskala** (Passagiere), nicht in der normierten 0–1-Skala.

In [ ]:
net.eval()
with torch.no_grad():
    vorhersage_norm = net(X_test)

# Rück-Normalisierung in die Originalskala (Passagierzahlen)
vorhersage = scaler.inverse_transform(vorhersage_norm.numpy())
y_test_original = scaler.inverse_transform(y_test.numpy())

mse = mean_squared_error(y_test_original, vorhersage)
mae = mean_absolute_error(y_test_original, vorhersage)
print(f"Test-MAE: {mae:.1f} Passagiere (im Schnitt daneben)")
print(f"Test-MSE: {mse:.1f}")


### 10 · Die Pipeline sichtbar machen

Jetzt das Herzstück – wie in Abschnitt 6 des 2D-Notebooks, nur als **Linienplots**
statt Bilder: Wir schicken **ein** Test-Fenster durchs trainierte Netz und zeigen
jede Stufe.

**Eingabe → Filter → Faltung → ReLU → MaxPool**

In [ ]:
def zeige_feature_maps(tensor, titel, farbe=MITTELBLAU):
    '''Zeigt alle Kanäle (Feature Maps) eines 1D-Tensors (1, K, T) als Linienplots.'''
    n = tensor.shape[1]
    fig, axs = plt.subplots(1, n, figsize=(1.6 * n, 1.6), sharey=False)
    for i in range(n):
        axs[i].plot(tensor[0, i], color=farbe)
        axs[i].set_title(f"#{i}", fontsize=8)
        axs[i].set_xticks([]); axs[i].set_yticks([])
    fig.suptitle(titel, color=DUNKELBLAU)
    plt.tight_layout(); plt.show()

idx = 0
fenster = X_test[idx:idx + 1]

net.eval()
with torch.no_grad():
    logits = net(fenster)
    stufen = {
        "Faltung (Conv1d)": net[:1](fenster),
        "nach ReLU":        net[:2](fenster),
        "nach MaxPool":     net[:3](fenster),
    }
pred_wert = scaler.inverse_transform(logits.numpy())[0, 0]
wahr_wert = scaler.inverse_transform(y_test[idx:idx + 1].numpy())[0, 0]

plt.figure(figsize=(4, 2))
plt.plot(fenster[0, 0], "o-", color=DUNKELBLAU)
plt.title(f"Eingabefenster – wahr: {wahr_wert:.0f}, vorhergesagt: {pred_wert:.0f}")
plt.tight_layout(); plt.show()

zeige_feature_maps(stufen["Faltung (Conv1d)"], "Schritt 1: Feature Maps nach der Faltung (Länge 12)")
zeige_feature_maps(stufen["nach ReLU"],        "Schritt 2: nach ReLU – negative Werte sind 0 (Länge 12)")
zeige_feature_maps(stufen["nach MaxPool"],     "Schritt 3: nach MaxPool – verkleinert auf Länge 6")


### 11 · Was hat das Netz gelernt?

### Filter vorher / nachher
Wie beim 2D-CNN sind die Filter zu Beginn reiner Zufall. Statt 3×3-Bildausschnitten
erkennen sie hier **Muster entlang der Zeit** – z. B. einen Anstieg oder ein
saisonales Auf und Ab über 3 Monate.

In [ ]:
kernel_nachher = net[0].weight.detach()
vmax = max(kernel_vorher.abs().max().item(), kernel_nachher.abs().max().item())

fig, axs = plt.subplots(2, CONV_FILTERS, figsize=(12, 2.6), sharey=True)
for i in range(CONV_FILTERS):
    axs[0, i].plot(kernel_vorher[i, 0],  color=MITTELBLAU); axs[0, i].axis("off")
    axs[1, i].plot(kernel_nachher[i, 0], color=DUNKELBLAU); axs[1, i].axis("off")
    axs[0, i].set_ylim(-vmax, vmax); axs[1, i].set_ylim(-vmax, vmax)
fig.text(0.5, 0.98, f"Filter vor und nach dem Training (je {KERNEL_SIZE} Zeitschritte)", ha="center", color=DUNKELBLAU, fontsize=12)
fig.text(0.085, 0.70, "vorher",  va="center", ha="right", color=DUNKELBLAU)
fig.text(0.085, 0.28, "nachher", va="center", ha="right", color=DUNKELBLAU)
plt.tight_layout(rect=[0.10, 0, 1, 0.92]); plt.show()


### 12 · Vorhersage & ehrlicher Vergleich

Der entscheidende Plot – wie in der App: die komplette Reihe mit echten Werten, und
darübergelegt die Modellvorhersage **nur auf den Testdaten**, die das Netz nie
gesehen hat. Die rote Linie markiert den Start der Testdaten.

In [ ]:
train_groesse = len(y_train)
zeitachse = np.arange(len(passagiere))
vorhersage_achse = np.arange(train_groesse + WINDOW_SIZE, len(passagiere))

plt.figure(figsize=(10, 4))
plt.plot(zeitachse, passagiere, color=MITTELBLAU, label="Echte Werte (gesamter Datensatz)")
plt.plot(vorhersage_achse, vorhersage, "--", color="red", label="Modellvorhersage (nur Testdaten)")
plt.axvline(x=train_groesse + WINDOW_SIZE, color="gray", linestyle=":", label="Start der Testdaten")
plt.title("1D-CNN – Vorhersage auf ungesehenen Testdaten")
plt.xlabel("Zeitindex (Monate)"); plt.ylabel("Passagiere (in 1000)")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
